# Armut Association Rule Learning Projesi

Armut müşteri verisi üzerinde birliktelik kuralı öğrenimi ile hizmet tavsiye sistemi çalışması.

## İş Problemi

Türkiye'nin en büyük online hizmet platformu olan Armut, hizmet verenler ile hizmet almak isteyenleri buluşturmaktadır.
Bilgisayarın veya akıllı telefonunun üzerinden birkaç dokunuşla temizlik, tadilat, nakliyat gibi hizmetlere kolayca
ulaşılmasını sağlamaktadır.

Hizmet alan kullanıcıları ve bu kullanıcıların almış oldukları servis ve kategorileri içeren veri setini kullanarak
Association Rule Learning ile ürün tavsiye sistemi oluşturulmak istenmektedir.

## Veri Seti

Veri seti müşterilerin aldıkları servislerden ve bu servislerin kategorilerinden oluşmaktadır.
Alınan her hizmetin tarih ve saat bilgisini içermektedir.

**Değişkenler**

- **UserId:** Müşteri numarası
- **ServiceId:** Her kategoriye ait anonimleştirilmiş servislerdir. (Örnek: Temizlik kategorisi altında koltuk yıkama servisi)
  Bir ServiceId farklı kategoriler altında bulunabilir ve farklı kategoriler altında farklı servisleri ifade eder.
  (Örnek: CategoryId'si 7 ServiceId'si 4 olan hizmet petek temizliği iken CategoryId'si 2 ServiceId'si 4 olan hizmet mobilya montaj)
- **CategoryId:** Anonimleştirilmiş kategorilerdir. (Örnek: Temizlik, nakliyat, tadilat kategorisi)
- **CreateDate:** Hizmetin satın alındığı tarih

## Görevler

1. **GÖREV 1:** Veriyi Hazırlama
2. **GÖREV 2:** Birliktelik Kuralları Üretiniz

In [1]:
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules

---
## GÖREV 1: Veriyi Hazırlama

### Adım 1: Veriyi Okuma

`armut_data.csv` dosyasını okutunuz.

In [2]:
df_ = pd.read_csv("datasets/armut_data.csv")
df = df_.copy()
df.head()

,UserId,ServiceId,CategoryId,CreateDate
0,25446,4,5,2017-08-06 16:11:00
1,22948,48,5,2017-08-06 16:12:00
2,10618,0,8,2017-08-06 16:13:00
3,7256,9,4,2017-08-06 16:14:00
4,25446,48,5,2017-08-06 16:16:00


### Adım 2: Hizmet Değişkeni Oluşturma

ServisID her bir CategoryID özelinde farklı bir hizmeti temsil etmektedir.
ServiceID ve CategoryID'yi `_` ile birleştirerek hizmetleri temsil edecek yeni bir değişken oluşturunuz.

In [3]:
df["Hizmet"] = df["ServiceId"].astype(str) + "_" + df["CategoryId"].astype(str)
df.head()

,UserId,ServiceId,CategoryId,CreateDate,Hizmet
0,25446,4,5,2017-08-06 16:11:00,4_5
1,22948,48,5,2017-08-06 16:12:00,48_5
2,10618,0,8,2017-08-06 16:13:00,0_8
3,7256,9,4,2017-08-06 16:14:00,9_4
4,25446,48,5,2017-08-06 16:16:00,48_5


### Adım 3: Sepet Tanımı Oluşturma

Veri seti hizmetlerin alındığı tarih ve saatten oluşmaktadır, herhangi bir sepet tanımı (fatura vb.) bulunmamaktadır.
Association Rule Learning uygulayabilmek için bir sepet (fatura vb.) tanımı oluşturulması gerekmektedir.

Burada sepet tanımı her bir müşterinin aylık aldığı hizmetlerdir. Örneğin; 7256 id'li müşteri 2017'in 8. ayında aldığı 9_4, 46_4 hizmetleri bir sepeti;
2017'nin 10. ayında aldığı 9_4, 38_4 hizmetleri başka bir sepeti ifade etmektedir. Sepetleri unique bir ID ile tanımlanması gerekmektedir.

Bunun için öncelikle sadece yıl ve ay içeren yeni bir date değişkeni oluşturunuz. UserID ve yeni oluşturduğunuz date değişkenini `_`
ile birleştirirek ID adında yeni bir değişkene atayınız.

In [ ]:
df["CreateDate"] = pd.to_datetime(df["CreateDate"])
df.head()

,UserId,ServiceId,CategoryId,CreateDate,Hizmet
0,25446,4,5,2017-08-06 16:11:00,4_5
1,22948,48,5,2017-08-06 16:12:00,48_5
2,10618,0,8,2017-08-06 16:13:00,0_8
3,7256,9,4,2017-08-06 16:14:00,9_4
4,25446,48,5,2017-08-06 16:16:00,48_5


In [5]:
df["New_Date"] = df["CreateDate"].dt.strftime("%Y-%m")
df.head()

,UserId,ServiceId,CategoryId,CreateDate,Hizmet,New_Date
0,25446,4,5,2017-08-06 16:11:00,4_5,2017-08
1,22948,48,5,2017-08-06 16:12:00,48_5,2017-08
2,10618,0,8,2017-08-06 16:13:00,0_8,2017-08
3,7256,9,4,2017-08-06 16:14:00,9_4,2017-08
4,25446,48,5,2017-08-06 16:16:00,48_5,2017-08


In [6]:
df["SepetID"] = df["UserId"].astype(str) + "_" + df["New_Date"]
df.head()

,UserId,ServiceId,CategoryId,CreateDate,Hizmet,New_Date,SepetID
0,25446,4,5,2017-08-06 16:11:00,4_5,2017-08,25446_2017-08
1,22948,48,5,2017-08-06 16:12:00,48_5,2017-08,22948_2017-08
2,10618,0,8,2017-08-06 16:13:00,0_8,2017-08,10618_2017-08
3,7256,9,4,2017-08-06 16:14:00,9_4,2017-08,7256_2017-08
4,25446,48,5,2017-08-06 16:16:00,48_5,2017-08,25446_2017-08


---
## GÖREV 2: Birliktelik Kuralları Üretiniz

### Adım 1: Sepet Hizmet Pivot Table Oluşturma

Aşağıdaki gibi sepet hizmet pivot table'i oluşturunuz.

```
Hizmet         0_8  10_9  11_11  12_7  13_11  14_7  15_1  16_8  17_5  18_4..
SepetID
0_2017-08        0     0      0     0      0     0     0     0     0     0..
0_2017-09        0     0      0     0      0     0     0     0     0     0..
0_2018-01        0     0      0     0      0     0     0     0     0     0..
0_2018-04        0     0      0     0      0     1     0     0     0     0..
10000_2017-08    0     0      0     0      0     0     0     0     0     0..
```

In [ ]:
# Adım 1: Önce sepet-hizmet ilişkisini gösterecek bir pivot tablo oluşturuyoruz.
# Bunun için, satırlarda 'SepetID', sütunlarda ise 'Hizmet' olacak şekilde bir pivot tablo oluşturuyoruz.
# 'values' parametresine 'UserId' veriyoruz, böylece her sepet-hizmet kombinasyonunda kaç kayıt olduğuna bakacağız.
# 'aggfunc' olarak 'count' kullandık, böylece her hücrede ilgili sepet-hizmet kombinasyonu kaç kere alınmış, onu gösterecek.
# Eksik olan değerleri ise fill_value=0 diyerek 0 ile dolduruyoruz.

pivot_table = df.pivot_table(
    index='SepetID',       # satır isimleri
    columns='Hizmet',      # sütun isimleri
    values='UserId',       # değer olarak UserId kullanılıyor
    aggfunc='count',       # aynı sepet-hizmet için tekrar varsa say
    fill_value=0           # eksik varsa 0 yaz
)

# Adım 2: Bu tablodaki değerler 0'dan büyükse '1' yapıyoruz, diğerleri zaten 0 olarak kalıyor.
# Böylece bir sepette bir hizmet varsa 1, yoksa 0 olacak şekilde binary bir matrix elde ediyoruz.

invoice_product_df = (pivot_table > 0).astype(int)

# Adım 3: Tabloyu görüntüleyerek son haline bakalım (ilk 5 satır)
invoice_product_df.head()

Hizmet,0_8,10_9,11_11,12_7,13_11,14_7,15_1,16_8,17_5,18_4,...,46_4,47_7,48_5,49_1,4_5,5_11,6_7,7_3,8_5,9_4
SepetID,,,,,,,,,,,,,,,,,,,,,
0_2017-08,0,0,0,0,0,0,0,0,0,0,...,1,0,1,0,0,0,0,0,0,0
0_2017-09,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,0,0,0,0,0
0_2018-01,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
0_2018-04,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
10000_2017-08,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,0,0,0,0


### Adım 2: Birliktelik Kurallarını Oluşturma

Birliktelik kurallarını oluşturunuz.

In [ ]:
# invoice_product_df'de her bir hizmetin ve kombinasyonlarının sepette bulunma oranlarına göre sık görülen (frequent) hizmet kümelerini buluyoruz.
freq_items = apriori(invoice_product_df, min_support=0.01, use_colnames=True)

# Bulunan sık hizmet kümeleri üzerinden ürün (hizmet) birliktelik kurallarını oluşturuyoruz.
# Burada "support" metriğini ve en az %1'lik bir eşik seçiyoruz.
rules = association_rules(freq_items, metric="support", min_threshold=0.01)

# Kuralların ilk birkaç satırını inceleyelim.
rules.head()

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/mlxtend/frequent_patterns/fpcommon.py:175: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({2_0}),frozenset({13_11}),0.130286,0.056627,0.012819,0.098394,1.737574,1.0,0.005442,1.046325,0.488074,0.073635,0.044274,0.162388
1,frozenset({13_11}),frozenset({2_0}),0.056627,0.130286,0.012819,0.226382,1.737574,1.0,0.005442,1.124216,0.449965,0.073635,0.110491,0.162388
2,frozenset({2_0}),frozenset({15_1}),0.130286,0.120963,0.033951,0.260588,2.154278,1.0,0.018191,1.188833,0.616073,0.156242,0.158839,0.270631
3,frozenset({15_1}),frozenset({2_0}),0.120963,0.130286,0.033951,0.280673,2.154278,1.0,0.018191,1.209066,0.609539,0.156242,0.172915,0.270631
4,frozenset({33_4}),frozenset({15_1}),0.027310,0.120963,0.011233,0.411311,3.400299,1.0,0.007929,1.493211,0.725728,0.081967,0.330302,0.252086


### Adım 3: Hizmet Önerisi

`arl_recommender` fonksiyonunu kullanarak en son `2_0` hizmetini alan bir kullanıcıya hizmet önerisinde bulununuz.

In [10]:
def arl_recommender(rules_df, product_id, rec_count=1):
    """
    Verilen birliktelik kuralları veri çerçevesi (rules_df) üzerinden,
    belirli bir ürün (product_id) için önerilecek ürünleri belirler.

    Fonksiyonun çalışma prensibi:
    - Kuralları "lift" değerine göre azalan şekilde sıralar.
    - "antecedents" (öncül) kümesinde product_id yer alan kuralların,
      "consequents" (sonuç/öneri) ürününü öneri listesine ekler.
    - Belirtilen sayıda (rec_count) öneriyi döndürür.

    Parametreler:
    rules_df : pd.DataFrame
        Birliktelik kurallarını içeren veri çerçevesi.
    product_id : int/str
        Sepette bulunan ve önerilere temel olacak ürün kodu.
    rec_count : int, optional
        Kaç adet öneri döndürüleceği (varsayılan: 1).

    Returns
    -------
    list
        Önerilen ürünlerin bir listesi.
    """
    sorted_rules = rules_df.sort_values("lift", ascending=False)
    recommendation_list = []
    for i, product in enumerate(sorted_rules["antecedents"]):
        for j in list(product):
            if j == product_id:
                recommendation_list.append(list(sorted_rules.iloc[i]["consequents"])[0])
    return recommendation_list[0:rec_count]

arl_recommender(rules, "2_0", 1)

['22_0']